# Brain Tumor MRI Classification using ResNet18

This notebook implements a deep learning pipeline for classifying brain MRI scans (Tumor vs. Non-Tumor) using a fine-tuned ResNet18 architecture. It includes automated data downloading, dynamic image preprocessing (cropping & CLAHE), cross-validation, and mixed-precision training.

## Table of Contents
1. [Environment Setup & Imports](#1-environment-setup--imports)
2. [Hyperparameters & Configuration](#2-hyperparameters--configuration)
3. [Data Preprocessing & Caching](#3-data-preprocessing--caching)
4. [Dataset Handlers & DataLoaders](#4-dataset-handlers--dataloaders)
5. [Model Architecture & Metrics](#5-model-architecture--metrics)
6. [Training & Evaluation (Main Loop)](#6-training--evaluation)

# ==========================================
# 1. Environment Setup & Imports
# ==========================================

In [22]:
%pip install kagglehub

In [23]:
# ==========================================
# 1. Environment Setup & Imports
# ==========================================
import os
import re
import sys
import copy
import json
import pickle
import random
import logging
from pathlib import Path

import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg") # Prevents display bugs in headless environments (like Colab)
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve
)

# Mount Google Drive if running in Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass # Not in Colab environment

logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
log = logging.getLogger(__name__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Hyperparameters & Configuration
Centralized configuration block to manage training hyperparameters, paths, and hardware settings.

In [24]:
# ==========================================
# 2. Configuration
# ==========================================
CONFIG = {
    # -- Training Settings --
    "num_epochs":         25,
    "batch_size":         32,     # Adjust based on GPU VRAM
    "lr":                 1e-4,   # Learning rate for AdamW
    "weight_decay":       1e-4,   # L2 regularization factor
    "n_folds":            5,      # Cross-validation folds
    "num_workers":        2,      # Subprocesses for data loading
    "seed":               42,     # For reproducibility
    
    # -- Paths --
    "out_dir":            "/content/drive/MyDrive/mybrain_research",
    "brisc_cache_path":    "brisc_cache.pkl",
    "figshare_cache_path": "figshare_cache.pkl",
    "v3_model_path":       "/content/drive/MyDrive/Brain_Tumor_Research/best_model_brisc.pth",
    
    # -- Execution Flags --
    "skip_training":      False,
    "slices_per_patient": 10,
}

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

# Device configuration (PyTorch specific)
# Automatically falls back to CPU if no GPU is detected
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info(f"Using Device: {device}")

## 3. Data Preprocessing & Caching
Medical images often contain artifacts or empty space. The `preprocess_image` function uses OpenCV to extract the region of interest and enhances contrast using CLAHE. We cache these results to RAM/Disk to prevent CPU bottlenecks during PyTorch's `DataLoader` fetching.

In [25]:
# ==========================================
# 3. Preprocessing
# ==========================================
def preprocess_image(image_path: str, output_size: int = 224) -> np.ndarray:
    """
    Reads, cleans, and standardizes an MRI scan.
    
    Args:
        image_path (str): Path to the image file.
        output_size (int): Target resolution (default: 224 for ResNet).
        
    Returns:
        np.ndarray: A preprocessed 3-channel RGB image array.
    """
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # 1. Auto-crop: Isolate the brain by finding the largest external contour
    _, thresh = cv2.threshold(img, 10, 255, cv2.THRESH_BINARY)
    kernel    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    thresh    = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        all_pts    = np.concatenate(contours)
        x, y, w, h = cv2.boundingRect(all_pts)
        pad        = 2
        img = img[max(0, y - pad): y + h + pad, max(0, x - pad): x + w + pad]

    # 2. CLAHE: Enhances local contrast to highlight tumor boundaries
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img   = clahe.apply(img)

    # 3. Normalization: Z-score scaling converted back to uint8
    f   = img.astype(np.float32)
    f   = (f - f.mean()) / (f.std() + 1e-8)
    f   = (f - f.min()) / (f.max() - f.min() + 1e-8)
    img = (f * 255).astype(np.uint8)

    # 4. Format for ResNet: Resize and convert 1-channel grayscale to 3-channel RGB
    img = cv2.resize(img, (output_size, output_size), interpolation=cv2.INTER_CUBIC)
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    return img

def build_cache(records: list, cache_path: str = "") -> dict:
    """ Processes all images upfront to prevent DataLoader bottlenecks during training. """
    if cache_path and os.path.exists(cache_path):
        log.info(f"Loading existing image cache from {cache_path}...")
        with open(cache_path, "rb") as f:
            return pickle.load(f)

    log.info(f"Building cache for {len(records)} images...")
    cache = {}
    for r in tqdm(records, desc="Preprocessing", unit="img"):
        img = preprocess_image(r["path"])
        # Fallback to zero-tensor if image is corrupted
        cache[r["path"]] = img if img is not None else np.zeros((224, 224, 3), dtype=np.uint8)

    if cache_path:
        with open(cache_path, "wb") as f:
            pickle.dump(cache, f)

    return cache

## 4. Dataset Handlers & DataLoaders
This section manages the PyTorch `Dataset` logic. We apply Torchvision transforms (augmentations) here. Augmentations are only applied to the training set to prevent the model from overfitting to specific image orientations.

In [ ]:
# ==========================================
# 4. Data Handlers & DataLoaders
# ==========================================

# Mappings and Regex for parsing complex medical filenames
BRISC_TYPE_MAP = {"gl": "glioma", "me": "meningioma", "pi": "pituitary", "no": "no_tumor"}
NEGATIVE_FOLDERS = {"no_tumor", "notumor", "no-tumor", "normal", "healthy", "negative"}
_BRISC_RE = re.compile(r'^brisc2025_(train|test)_(\d{5})_([a-z]{2})_(?:ax|co|sa)_t1', re.IGNORECASE)

def parse_brisc_filename(filename: str, slices_per_patient: int):
    """Extracts metadata and ensures patient-level splitting from BRISC filenames."""
    stem = Path(filename).stem
    m    = _BRISC_RE.match(stem)
    if not m:
        return None

    slice_id  = int(m.group(2))
    type_code = m.group(3).lower()
    label     = 0 if type_code == "no" else 1

    # Grouping slices to prevent data leakage across folds
    patient_bucket = slice_id // slices_per_patient
    patient_id     = f"{type_code}_{patient_bucket:04d}"

    return {
        "slice_id":   slice_id,
        "type_code":  type_code,
        "type_name":  BRISC_TYPE_MAP.get(type_code, type_code),
        "label":      label,
        "patient_id": patient_id,
        "split":      m.group(1).lower(),
    }

def load_brisc(brisc_root: str, slices_per_patient: int = 10):
    """Traverses the BRISC directory and builds the dataset manifest."""
    class_root = os.path.join(brisc_root, "brisc2025", "classification_task")
    if not os.path.isdir(class_root):
        class_root = os.path.join(brisc_root, "classification_task")

    records, skipped = [], 0

    for dirpath, _, filenames in os.walk(class_root):
        for fname in sorted(filenames):
            if Path(fname).suffix.lower() not in IMG_EXTS:
                continue
            parsed = parse_brisc_filename(fname, slices_per_patient)
            if parsed is None:
                skipped += 1
                continue

            records.append({
                "path":       os.path.join(dirpath, fname),
                "label":      parsed["label"],
                "patient_id": parsed["patient_id"],
                "type_name":  parsed["type_name"],
                "slice_id":   parsed["slice_id"],
            })

    log.info(f"BRISC Dataset loaded: Found {len(records)} images.")
    return records

def load_figshare(figshare_root: str):
    """Traverses the Figshare directory. Labels are determined by folder names."""
    records = []
    for dirpath, _, filenames in os.walk(figshare_root):
        class_folder = os.path.basename(dirpath)
        if class_folder.lower() in {"training", "testing", "train", "test", "brain-tumor-mri-dataset", ""}:
            continue

        label = 0 if class_folder.lower() in NEGATIVE_FOLDERS else 1

        for fname in sorted(filenames):
            if Path(fname).suffix.lower() not in IMG_EXTS:
                continue
            full_path = os.path.join(dirpath, fname)

            # Generate a pseudo patient ID
            tokens = re.findall(r'\d+', Path(fname).stem)
            pid = f"fg_{tokens[-1].zfill(6)}" if tokens else f"fg_{abs(hash(Path(fname).stem)) % 1_000_000:06d}"

            records.append({"path": full_path, "label": label, "patient_id": pid})

    log.info(f"Figshare Dataset loaded: Found {len(records)} images.")
    return records

def build_transforms(augment: bool) -> T.Compose:
    """Constructs PyTorch transformation pipelines with ImageNet standards."""
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD  = [0.229, 0.224, 0.225]

    ops = [T.ToPILImage()]

    if augment:
        ops += [
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.3),
            T.RandomRotation(degrees=15),
            T.ColorJitter(brightness=0.15, contrast=0.15),
            T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
        ]

    ops += [
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
    return T.Compose(ops)

class BrainMRIDataset(Dataset):
    """PyTorch dataset wrapper fetching images from disk or cache."""
    def __init__(self, records, transform, cache=None):
        self.records   = records
        self.transform = transform
        self.cache     = cache

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx: int):
        rec = self.records[idx]
        path = rec["path"]

        if self.cache is not None and path in self.cache:
            img = self.cache[path]
        else:
            img = preprocess_image(path)
            if img is None: img = np.zeros((224, 224, 3), dtype=np.uint8)

        return self.transform(img), rec["label"]

## 5. DANN Architecture & Metrics
We are adapting a pre-trained ResNet18 into a Domain-Adversarial Neural Network. 
The model now splits into two heads:
1. **Class Predictor:** Predicts Tumor vs. Non-Tumor (using source data).
2. **Domain Predictor:** Predicts BRISC vs. Figshare (using source + target data).

A Gradient Reversal Layer (GRL) sits before the Domain Predictor to force the feature extractor to learn domain-invariant features.

In [31]:
# ==========================================
# 5. DANN Model & Metrics
# ==========================================
class GradientReversal(torch.autograd.Function):
    """
    The magic behind DANN. 
    Forward pass: Acts as an identity function.
    Backward pass: Reverses gradients by multiplying them by -alpha.
    """
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        output = grad_output.neg() * ctx.alpha
        return output, None

class DANN_ResNet18(nn.Module):
    def __init__(self, pretrained: bool = True):
        super(DANN_ResNet18, self).__init__()
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.resnet18(weights=weights)
        
        # 1. Feature Extractor: ResNet without the final fully connected layer
        self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1])
        in_feat = backbone.fc.in_features
        
        # 2. Class Classifier Head (Tumor vs No-Tumor)
        self.class_classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(in_feat, 1)
        )
        
        # 3. Domain Classifier Head (Source vs Target Dataset)
        self.domain_classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(in_feat, 256),
            nn.ReLU(True),
            nn.Dropout(p=0.5),
            nn.Linear(256, 1)
        )

    def forward(self, x, alpha=None):
        # Extract features (B, 512, 1, 1) -> Flatten to (B, 512)
        features = self.feature_extractor(x)
        features = features.view(features.size(0), -1)
        
        # Always compute the class prediction
        class_output = self.class_classifier(features)
        
        # If alpha is provided (during training), compute domain prediction too
        if alpha is not None:
            reverse_features = GradientReversal.apply(features, alpha)
            domain_output = self.domain_classifier(reverse_features)
            return class_output, domain_output
            
        return class_output # Inference mode (only class matters)

def compute_metrics(labels: np.ndarray, probs: np.ndarray, threshold: float = 0.5) -> dict:
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()

    return {
        "accuracy":    accuracy_score(labels, preds),
        "precision":   precision_score(labels, preds, zero_division=0),
        "recall":      recall_score(labels, preds, zero_division=0),
        "specificity": float(tn / (tn + fp + 1e-8)),
        "f1":          f1_score(labels, preds, zero_division=0),
        "auc_roc":     roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else 0.0,
    }

## 6. DANN Training & Evaluation
The training loop now takes **two** DataLoaders simultaneously: the source (BRISC) and the target (Figshare). We use `itertools.cycle` to constantly loop through the unlabeled target dataset alongside our labeled source dataset.

In [28]:
# ==========================================
# 6. DANN Main Execution
# ==========================================
import itertools

def train_dann_epoch(model, source_loader, target_loader, optimizer, scaler, device, current_epoch, total_epochs):
    model.train()
    loss_sum, correct_class, total = 0.0, 0, 0
    
    class_criterion = nn.BCEWithLogitsLoss()
    domain_criterion = nn.BCEWithLogitsLoss()

    # Cycle the target loader so we never run out of target images during an epoch
    target_iter = itertools.cycle(target_loader)
    len_dataloader = len(source_loader)

    for i, (src_imgs, src_labels) in enumerate(source_loader):
        tgt_imgs, _ = next(target_iter)
        
        src_imgs = src_imgs.to(device, non_blocking=True)
        src_labels = src_labels.float().unsqueeze(1).to(device, non_blocking=True)
        tgt_imgs = tgt_imgs.to(device, non_blocking=True)

        # Dynamic Alpha Schedule: Starts at 0 and slowly scales to 1 
        # Prevents the noisy early domain classifier from ruining the feature extractor
        p = float(i + current_epoch * len_dataloader) / (total_epochs * len_dataloader)
        alpha = 2. / (1. + np.exp(-10 * p)) - 1
        
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
            # --- 1. Source Domain Pass ---
            src_class_logits, src_domain_logits = model(src_imgs, alpha)
            src_class_loss = class_criterion(src_class_logits, src_labels)
            
            # Domain 0 = Source
            src_domain_labels = torch.zeros_like(src_domain_logits).to(device)
            src_domain_loss = domain_criterion(src_domain_logits, src_domain_labels)
            
            # --- 2. Target Domain Pass ---
            _, tgt_domain_logits = model(tgt_imgs, alpha)
            
            # Domain 1 = Target
            tgt_domain_labels = torch.ones_like(tgt_domain_logits).to(device)
            tgt_domain_loss = domain_criterion(tgt_domain_logits, tgt_domain_labels)
            
            # --- 3. Combined Loss ---
            total_loss = src_class_loss + src_domain_loss + tgt_domain_loss

        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_sum += total_loss.item() * src_imgs.size(0)
        preds = (torch.sigmoid(src_class_logits) >= 0.5).long()
        correct_class += (preds == src_labels.long()).sum().item()
        total += src_imgs.size(0)

    return loss_sum / total, correct_class / total

@torch.no_grad() 
def evaluate(model, loader, device):
    """Evaluates the model without passing alpha (ignores domain classification)."""
    model.eval() 
    criterion = nn.BCEWithLogitsLoss()
    loss_sum, correct, total = 0.0, 0, 0
    all_labels, all_probs = [], []

    for imgs, labels in loader:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.float().unsqueeze(1).to(device, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
            logits = model(imgs) # No alpha passed, returns only class_logits
            loss   = criterion(logits, labels)

        probs     = torch.sigmoid(logits)
        loss_sum += loss.item() * imgs.size(0)
        correct  += ((probs >= 0.5).long() == labels.long()).sum().item()
        total    += imgs.size(0)

        all_labels.extend(labels.cpu().numpy().flatten())
        all_probs.extend(probs.cpu().numpy().flatten())

    return (loss_sum / total, correct / total, np.array(all_labels), np.array(all_probs))

## 7. Cross-Validation & Execution
Executing the DANN pipeline. We modify the Cross-Validation runner to inject the unlabeled target dataset into the training loop.

In [29]:
def run_dann_cv(brisc_records, figshare_records, device, config, brisc_cache=None, figshare_cache=None):
    labels_arr = np.array([r["label"] for r in brisc_records])
    groups_arr = np.array([r["patient_id"] for r in brisc_records])

    sgkf = StratifiedGroupKFold(n_splits=config["n_folds"], shuffle=True, random_state=config["seed"])
    
    # Setup Target (Figshare) DataLoader for Domain Adaptation
    # We only use transforms, labels are ignored during training by the DANN
    target_ds = BrainMRIDataset(figshare_records, build_transforms(augment=True), cache=figshare_cache)
    target_dl = DataLoader(target_ds, batch_size=config["batch_size"], shuffle=True, num_workers=config["num_workers"], drop_last=True)

    fold_results = []
    best_auc, best_model = -1.0, None

    for fold, (tr_idx, vl_idx) in enumerate(sgkf.split(brisc_records, labels_arr, groups=groups_arr), start=1):
        log.info(f"\n{'='*58}\n  DANN FOLD {fold}/{config['n_folds']} \n{'='*58}")

        train_recs = [brisc_records[i] for i in tr_idx]
        val_recs   = [brisc_records[i] for i in vl_idx]

        train_ds = BrainMRIDataset(train_recs, build_transforms(augment=True), cache=brisc_cache)
        val_ds   = BrainMRIDataset(val_recs, build_transforms(augment=False), cache=brisc_cache)

        train_dl = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True, num_workers=config["num_workers"], drop_last=True)
        val_dl   = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False, num_workers=config["num_workers"])

        model     = DANN_ResNet18().to(device)
        optimizer = optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["num_epochs"])
        scaler = torch.amp.GradScaler('cuda', enabled=(device.type == "cuda"))

        best_vl_auc, best_wts = -1.0, None

        for epoch in range(config["num_epochs"]):
            # Pass BOTH source and target loaders
            tl, ta = train_dann_epoch(model, train_dl, target_dl, optimizer, scaler, device, epoch, config["num_epochs"])
            val_loss, val_acc, val_labels, val_probs = evaluate(model, val_dl, device)
            scheduler.step()

            ep_auc = roc_auc_score(val_labels, val_probs) if len(np.unique(val_labels)) > 1 else 0.0
            log.info(f"  Ep {epoch+1:02d}/{config['num_epochs']} | TrLoss={tl:.4f} TrAcc={ta:.4f} | ValLoss={val_loss:.4f} ValAUC={ep_auc:.4f}")

            if ep_auc > best_vl_auc:
                best_vl_auc = ep_auc
                best_wts    = copy.deepcopy(model.state_dict())

        # Test the best weights for this fold
        model.load_state_dict(best_wts)
        _, _, fl, fp = evaluate(model, val_dl, device)
        m = compute_metrics(fl, fp)
        m["fold"] = fold
        fold_results.append(m)

        if m["auc_roc"] > best_auc:
            best_auc   = m["auc_roc"]
            best_model = copy.deepcopy(model)
            torch.save(best_wts, os.path.join(config["out_dir"], "best_dann_brisc.pth"))
            log.info(f"  ✓ New best DANN model saved (AUC={best_auc:.4f})")

    return best_model, fold_results

In [32]:
# ==========================================
# 8. Master Execution (Trigger Training)
# ==========================================
def main():
    # Setup seeds so we can reproduce our results exactly next time
    os.makedirs(CONFIG["out_dir"], exist_ok=True)
    random.seed(CONFIG["seed"])
    np.random.seed(CONFIG["seed"])
    torch.manual_seed(CONFIG["seed"])
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(CONFIG["seed"])

    # 1. Download Data via KaggleHub
    log.info("Downloading datasets...")
    import kagglehub
    brisc_path = kagglehub.dataset_download("briscdataset/brisc2025")
    figshare_path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

    # Parse the downloaded folders using the functions from your Data Handlers
    brisc_records    = load_brisc(brisc_path, slices_per_patient=CONFIG["slices_per_patient"])
    figshare_records = load_figshare(figshare_path)

    if not brisc_records or not figshare_records:
        log.error("Something went wrong, missing records. Check dataset paths.")
        return

    # 2. Build or load preprocessing cache
    log.info("\n── Checking Preprocessing Caches ──")
    brisc_cache    = build_cache(brisc_records, cache_path=CONFIG["brisc_cache_path"])
    figshare_cache = build_cache(figshare_records, cache_path=CONFIG["figshare_cache_path"])

    # 3. Start the DANN Training Pipeline
    if CONFIG.get("skip_training") and os.path.exists(CONFIG["v3_model_path"]):
        log.info(f"\n>>> Skipping Training. Model already exists at {CONFIG['v3_model_path']} <<<")
    else:
        log.info("\n>>> Starting DANN 5-Fold Cross Validation <<<")
        best_model, cv_results = run_dann_cv(
            brisc_records, 
            figshare_records, 
            device, 
            CONFIG, 
            brisc_cache=brisc_cache, 
            figshare_cache=figshare_cache
        )
        
        log.info(f"\nPipeline complete! Best DANN model saved to -> {CONFIG['out_dir']}/best_dann_brisc.pth")

# Trigger the script
if __name__ == "__main__":
    main()

Using Colab cache for faster access to the 'brisc2025' dataset.


KeyboardInterrupt: 